# Projeto RAG Avançado com Qdrant local + Maritaca `sabiazinho-4`

Este notebook implementa um **RAG Avançado** em cima da base `quikr_car.csv`, utilizando **ESTRITAMENTE AS MESMAS BIBLIOTECAS E MODELOS** do RAG simples, mas com arquitetura algorítmica melhorada.

As melhorias são:
1. **Query Expansion (Expansão de Consulta):** A LLM reescreve a pergunta do usuário para abranger diferentes vocabulários.


In [10]:
import os
import time
from pathlib import Path
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client import models
from openai import OpenAI


In [11]:
CSV_PATH = Path(os.getenv("CSV_PATH", r"./Data/quikr_car.csv"))

# Coleção para a versão avançada
COLLECTION_NAME = "quikr_car_rag_avancado"
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")

# O MESMO Modelo de Embedding do RAG Simples
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

MARITACA_MODEL_NAME = "sabiazinho-4"
MARITACA_BASE_URL = "https://chat.maritaca.ai/api"

# Vamos buscar 10 documentos por variação e unir os rankings.
TOP_K_PER_QUERY = 10
TOP_K_FINAL = 5
BATCH_SIZE = 64



In [12]:
load_dotenv()
MARITACA_API_KEY = os.getenv("MARITACA_API_KEY")

if not MARITACA_API_KEY:
    raise ValueError("Variável MARITACA_API_KEY não encontrada.")

qdrant_client = QdrantClient(url=QDRANT_URL)
maritaca_client = OpenAI(api_key=MARITACA_API_KEY, base_url=MARITACA_BASE_URL)



In [13]:
df = pd.read_csv(CSV_PATH)
df_clean = df.copy()
df_clean.columns = [col.strip() for col in df_clean.columns]
df_clean = df_clean.dropna(subset=["name", "company"]).fillna("").reset_index(drop=True)

def safe_get(row, column_name, default=""):
    return row[column_name] if column_name in row else default

def build_car_text(row):
    name = safe_get(row, "name")
    company = safe_get(row, "company")
    year = safe_get(row, "year")
    price = safe_get(row, "Price")
    kms = safe_get(row, "kms_driven")
    fuel = safe_get(row, "fuel_type")
    
    
    return f"Carro: {name}. Marca: {company}. Ano: {year}. Preço: {price}. Quilometragem: {kms}. Combustível: {fuel}."

df_clean["rag_text"] = df_clean.apply(build_car_text, axis=1)


In [14]:
print("Carregando modelo de embeddings...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
VECTOR_SIZE = embedding_model.get_sentence_embedding_dimension()



Carregando modelo de embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

C:\Users\nicol\AppData\Local\Temp\ipykernel_24428\614720732.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  VECTOR_SIZE = embedding_model.get_sentence_embedding_dimension()


In [15]:
existing_collections = [c.name for c in qdrant_client.get_collections().collections]
if COLLECTION_NAME in existing_collections:
    qdrant_client.delete_collection(collection_name=COLLECTION_NAME)

qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE)
)

def build_payload(row):
    return {
        "rag_text": str(safe_get(row, "rag_text")),
        "name": str(safe_get(row, "name")),
        "company": str(safe_get(row, "company")),
        "year": str(safe_get(row, "year")),
        "price": str(safe_get(row, "Price")),
        "kms_driven": str(safe_get(row, "kms_driven")),
        "fuel_type": str(safe_get(row, "fuel_type")),

    }

total_rows = len(df_clean)
for start_index in tqdm(range(0, total_rows, BATCH_SIZE), desc="Enviando ao Qdrant"):
    end_index = min(start_index + BATCH_SIZE, total_rows)
    batch_df = df_clean.iloc[start_index:end_index]
    batch_texts = batch_df["rag_text"].tolist()
    
    batch_vectors = embedding_model.encode(batch_texts, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)
    points = []
    
    for local_pos, (_, row) in enumerate(batch_df.iterrows()):
        global_index = start_index + local_pos
        points.append(models.PointStruct(
            id=int(global_index),
            vector=batch_vectors[local_pos].tolist(),
            payload=build_payload(row)
        ))
    qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points)



Enviando ao Qdrant:   0%|          | 0/14 [00:00<?, ?it/s]

## Célula 8 — Query Expansion (Expansão de Consulta)


In [16]:
def generate_query_variations(question):
    system_msg = "Você é um assistente especialista em buscas de carros. Gere 2 variações ou reescritas úteis da pergunta do usuário para ajudar em uma busca semântica em banco vetorial. Devolva apenas as variações separadas por quebra de linha, sem qualquer outro texto."
    
    try:
        response = maritaca_client.responses.create(
            model=MARITACA_MODEL_NAME,
            input=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": question}
            ],
            max_output_tokens=150,
            temperature=0.7
        )
        variations_text = response.output[0].content[0].text
        # Separa por linha e remove vazias
        variations = [v.strip() for v in variations_text.split('\n') if v.strip()]
        return variations
    except Exception as e:
        print(f"Erro na expansão de query: {e}")
        return []

# Testando a expansão
print("Teste Expansão:")


Teste Expansão:


## Célula 9 — Busca Avançada com RRF (Reciprocal Rank Fusion)


In [17]:
def search_cars_rrf(question, top_k_per_query=TOP_K_PER_QUERY, top_k_final=TOP_K_FINAL):
    # 1. Obter queries
    queries = [question]
    variations = generate_query_variations(question)
    queries.extend(variations)
    
    # 2. Guardar ranking de cada query
    # rrf_scores vai guardar: rrf_scores[doc_id] = nota_acumulada_rrf
    rrf_scores = {}
    # doc_payloads vai guardar os dados do filme para não perdermos a info
    doc_payloads = {}
    
    # 3. Fazer a busca de cada query no Qdrant
    for q in queries:
        q_vector = embedding_model.encode(q, convert_to_numpy=True, normalize_embeddings=True).tolist()
        
        try:
            points = qdrant_client.query_points(collection_name=COLLECTION_NAME, query=q_vector, limit=top_k_per_query, with_payload=True).points
        except AttributeError:
            points = qdrant_client.search(collection_name=COLLECTION_NAME, query_vector=q_vector, limit=top_k_per_query, with_payload=True)
            
        # 4. Calcular o RRF para os resultados desta query
        # rank começa em 1 para o primeiro resultado
        for rank, p in enumerate(points, start=1):
            doc_id = p.id
            if doc_id not in rrf_scores:
                rrf_scores[doc_id] = 0.0
                doc_payloads[doc_id] = p.payload
            
            # Fórmula padrão do RRF (k constante de 60)
            rrf_scores[doc_id] += 1.0 / (60.0 + rank)
            
    # 5. Ordenar pelo score RRF acumulado
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    
    # 6. Pegar apenas o Top K Final
    top_docs = sorted_docs[:top_k_final]
    
    # Reformatar para ficar fácil de usar no resto do código
    final_results = []
    for doc_id, score in top_docs:
        final_results.append({
            "id": doc_id,
            "rrf_score": score,
            "payload": doc_payloads[doc_id]
        })
        


In [18]:
def ask_maritaca_advanced(question, context):
    system_message = (
        "Você é um assistente de RAG avançado focado em anúncios de carros. "
        "Responda em português do Brasil de forma clara e objetiva. "
        "Use SOMENTE o contexto fornecido. Se não houver informação, diga que não sabe."
    )
    user_message = f"CONTEXTO RECUPERADO:\n{context}\n\nPERGUNTA: {question}"
    
    response = maritaca_client.responses.create(
        model=MARITACA_MODEL_NAME,
        input=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ],
        max_output_tokens=500,
        temperature=0.2,
    )
    return response.output[0].content[0].text

def build_context_from_advanced_results(results):
    blocks = []
    for i, doc in enumerate(results, start=1):
        rag_text = doc["payload"].get("rag_text", "")
        score = doc["rrf_score"]
        blocks.append(f"[Documento {i} | RRF Score: {score:.4f}]\n{rag_text}")
    return "\n\n---\n\n".join(blocks)

def chat_rag_avancado():
    print("= Chat RAG Avançado Iniciado =")
    print("Digite 'sair' para encerrar.\n")
    
    while True:
        question = input("Você: ")
        if question.strip().lower() == "sair":
            print("Chat encerrado.")
            break
            
        print("\n> Gerando expansões de query e buscando...")
        start_time = time.time()
        
        # Recupera os documentos fundidos pelo RRF
        results, queries_used = search_cars_rrf(question)
        
        print(f"\n[INFO] Queries pesquisadas em paralelo:")
        for q in queries_used:
            print(f"  - {q}")
            
        print("\n[INFO] Top Documentos Vencedores (via RRF):")
        for i, doc in enumerate(results, 1):
            title = doc["payload"].get("name")
            score = doc["rrf_score"]
            print(f"  {i}. {title} (Pontuação RRF acumulada: {score:.4f})")
            
        context = build_context_from_advanced_results(results)
        
        print("\n> Gerando resposta com Maritaca...")
        answer = ask_maritaca_advanced(question, context)
        
        print("\n" + "="*80)
        print(f"Assistente: {answer}")
        print("="*80 + "\n")
        
# Descomente a linha abaixo para testar no notebook:
